In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.rebuild_comparison import build_rebuild_comparison_stage

from utils.core.env_helpers import (
    env_float, 
    env_int, 
    env_str,
)

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

FLOAT_TOLERANCE = env_float(
    "SYNTHETIC_REBUILD_COMPARISON_FLOAT_TOLERANCE",
    1e-9,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

PREMELT_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

REBUILT_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

TARGET_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILD_COMPARISON_TABLE",
    "synthetic_sensor_rebuild_comparison_stage",
)

print("Rebuild comparison config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("premelt table:", PREMELT_SOURCE_TABLE_NAME)
print("rebuilt table:", REBUILT_DESTINATION_TABLE_NAME)
print("target table:", TARGET_TABLE_NAME)

In [ ]:
engine = get_engine_from_env()

----

In [ ]:
comparison_result = build_rebuild_comparison_stage(
    engine=engine,
    schema=SCHEMA,
    premelt_table=PREMELT_SOURCE_TABLE_NAME,
    rebuilt_table=REBUILT_DESTINATION_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    float_tolerance=FLOAT_TOLERANCE,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(comparison_result)

----

In [ ]:
summary_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS comparison_row_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = TRUE) AS all_match_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = FALSE) AS mismatch_count,
        MAX(comparison_mismatch_count) AS max_mismatch_count
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    """,
)

display(summary_dataframe)

----

In [ ]:
mismatch_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        comparison_mismatch_count,
        comparison_notes,
        exists_in_original,
        exists_in_rebuilt,
        rebuilt__rebuild_sensor_count,
        rebuilt__rebuild_is_complete
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE comparison_all_fields_match = FALSE
    ORDER BY observation_index
    LIMIT 50
    """,
)

display(mismatch_dataframe)

-----

In [ ]:
observation_to_check = 1

detail_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE observation_index = {int(observation_to_check)}
    """,
)

display(detail_dataframe)

----

In [ ]:
# Dispose Engine
engine.dispose()